# Evaluation Script: Mixed Content Enforcement Gaps

This script produces results for Section 5.5 Mixed Content Enforcement Gaps

### Connect to the MongoDB

In [ ]:
import tqdm
import pymongo

from urllib.parse import urlparse

client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["webview"]
dynamic_api_calls_collection = db["dynamic_api_calls"]

def print_latex_macro(name: str, value: str):
    print(f"\\newcommand{{\\{name}}}{{{value}}}")

### Get stats on the usage of different WebView APIs

Produces results for "API Usage"

In [2]:
amount_apps_using_webview = len(dynamic_api_calls_collection.distinct("source_package_name"))
apps_loading_data_api = dynamic_api_calls_collection.distinct("source_package_name", {"api": "LOAD_DATA"})
apps_loading_data_url = dynamic_api_calls_collection.distinct("source_package_name", {"api": "LOAD_URL", "params.0": {"$regex": "^data:"}})
apps_loading_data_with_base_url = dynamic_api_calls_collection.distinct("source_package_name", {"api": "LOAD_DATA_WITH_BASE_URL"})
apps_loading_file_url = dynamic_api_calls_collection.distinct("source_package_name", {"api": "LOAD_URL", "params.0": {"$regex": "^file:"}})

print_latex_macro("amountAppsLoadingDataWithAPI", f"{len(apps_loading_data_api):,}")
print_latex_macro("amountAppsLoadingDataWithURL", f"{len(apps_loading_data_url):,}")
print_latex_macro("amountAppsLoadingDataWithAPIPercent", f"{(len(apps_loading_data_api)/amount_apps_using_webview)*100:.2f}")
print_latex_macro("amountAppsLoadingDataWithURLPercent", f"{(len(apps_loading_data_url)/amount_apps_using_webview)*100:.2f}")
print_latex_macro("amountAppsLoadingDataWithBaseURL", f"{len(apps_loading_data_with_base_url):,}")
print_latex_macro("amountAppsLoadingDataWithBaseURLPercent", f"{(len(apps_loading_data_with_base_url)/amount_apps_using_webview)*100:.2f}")
print_latex_macro("amountAppsLoadingFileURL", f"{len(apps_loading_file_url):,}")
print_latex_macro("amountAppsLoadingFileURLPercent", f"{(len(apps_loading_file_url)/amount_apps_using_webview)*100:.2f}")

# 

\newcommand{\amountAppsLoadingDataWithAPI}{3,325}
\newcommand{\amountAppsLoadingDataWithURL}{26}
\newcommand{\amountAppsLoadingDataWithAPIPercent}{13.20}
\newcommand{\amountAppsLoadingDataWithURLPercent}{0.10}
\newcommand{\amountAppsLoadingDataWithBaseURL}{13,008}
\newcommand{\amountAppsLoadingDataWithBaseURLPercent}{51.65}
\newcommand{\amountAppsLoadingFileURL}{4,397}
\newcommand{\amountAppsLoadingFileURLPercent}{17.46}


#### Retrieve the commonly used base URLs in `loadDataWithBaseURL`

Produces results for "Commonly Used Base URLS"

In [3]:
data_with_base_url = dynamic_api_calls_collection.find({
    "api": "LOAD_DATA_WITH_BASE_URL"
})

amount_data_with_base_url = dynamic_api_calls_collection.count_documents({"api": "LOAD_DATA_WITH_BASE_URL"})

schemes = {}
items = []
for r in tqdm.tqdm(data_with_base_url, total=amount_data_with_base_url, desc="Retrieving base URLs"):
    package_name = r["source_package_name"]
    params = r["params"]
    base_url = params[0]
    parsed_url = urlparse(base_url)
    scheme = parsed_url.scheme
    netloc = parsed_url.netloc
    path = parsed_url.path
    
    if (scheme, netloc, path, base_url) not in schemes:
        schemes[(scheme, netloc, path, base_url)] = set()
    schemes[(scheme, netloc, path, base_url)].add(package_name)


schemes = dict(sorted(schemes.items(), key=lambda item: len(item[1]), reverse=True))    

all_but_https = set()
# do by protocol
top_protocols = {}
for (scheme, netloc, path, base_url), package_names in schemes.items():
    base_url_scheme = scheme
    if not base_url:
        base_url_scheme = "<None>"
    elif base_url_scheme == "":
        base_url_scheme = "<empty string>"
    elif base_url == "about:blank" or base_url == "about:blank/":
        base_url_scheme = "about:blank"
    elif base_url.startswith("about:"):
        base_url_scheme = base_url
    if base_url_scheme not in top_protocols:
        top_protocols[base_url_scheme] = set()
    top_protocols[base_url_scheme].update(package_names)


    if not base_url == "https":
        all_but_https.update(package_names)
    
# Sort and take top 10
sorted_top_protocols = sorted(top_protocols.items(), key=lambda x: len(x[1]), reverse=True)
for protocol, package_names in sorted_top_protocols:
    print(f"%{protocol}: {len(package_names)}")
    
print_latex_macro("appsUsingAboutBlank", f"{len(top_protocols["about:blank"]):,}")
print_latex_macro("appsUsingHTTPS", f"{len(top_protocols["https"]):,}")
print_latex_macro("appsUsingHTTP", f"{len(top_protocols["http"]):,}")
print_latex_macro("appsUsingFile", f"{len(top_protocols["file"]):,}")
print_latex_macro("appsUsingNone", f"{len(top_protocols["<None>"]):,}")
print_latex_macro("appsNonHTTPSPercentage", f"{len(all_but_https) / amount_apps_using_webview * 100:.2f}")



Retrieving base URLs: 100%|██████████| 259353/259353 [05:04<00:00, 850.62it/s] 

%https: 10140
%about:blank: 6705
%file: 3460
%<None>: 1863
%<empty string>: 1090
%http: 652
%blarg: 3
%fake: 3
%pb: 1
%x-data: 1
%content: 1
%ap: 1
%chrome: 1
%inline: 1
\newcommand{\appsUsingAboutBlank}{6,705}
\newcommand{\appsUsingHTTPS}{10,140}
\newcommand{\appsUsingHTTP}{652}
\newcommand{\appsUsingFile}{3,460}
\newcommand{\appsUsingNone}{1,863}
\newcommand{\appsNonHTTPSPercentage}{51.65}
